# Testies

### Importing packages

In [95]:
import numpy as np
from scipy.optimize import minimize

### Simulating AR(1)

In [ ]:
def simulate(T: int, beta: np.ndarray, sigma: np.ndarray, P: np.ndarray, seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    
    """
    This is a simulation function for an rs ar 
    """

    np.random.seed(seed)
    states = np.zeros(T, dtype=int)
    y = np.zeros(T)
    states[0] = np.random.choice([0, 1])
    y[0] = np.random.normal()

    for t in range(1, T):
        states[t] = np.random.choice([0, 1], p=P[states[t-1]])
        s = states[t]
        y[t] = beta[s]*y[t-1] + np.random.normal(scale=sigma[s])
    
    return y, states

### Run it

In [97]:
T = 100
beta = np.array([0.2,  0.8])
sigma = np.array([0.5, 1.0])
P = np.array([
    [0.95, 0.05],
    [0.05, 0.95]
])

y, states = simulate(T, beta, sigma, P, seed=42)

print(f"y shape: {y.shape}")
print(f"states shape: {states.shape}")
print(f"unique states: {np.unique(states)}")
print(f"first 5 y-values: {y[:5]}")
print(f"first 30 states: {states[:30]}")

assert len(y) == T
assert len(states) == T
assert set(np.unique(states)).issubset({0, 1})

y shape: (100,)
states shape: (100,)
unique states: [0 1]
first 5 y-values: [-0.55023449  0.14766964 -0.42887949 -0.14784949  0.16671998]
first 30 states: [0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1]


### Transforming

In [98]:
def transform(theta: np.ndarray) -> tuple: 
    """
    Transforming function
    """
    b1, b2, eta1, eta2, a1, a2 = theta
    
    beta1 = (1-np.exp(-b1)) / (1+np.exp(-b1))
    beta2 = (1-np.exp(-b2)) / (1+np.exp(-b2))

    sigma1 = np.exp(eta1)
    sigma2 = np.exp(eta2)

    p11 = 1 / (1+np.exp(-a1))
    p22 = 1 / (1+np.exp(-a2))

    return beta1, beta2, sigma1, sigma2, p11, p22

### Running it 

In [99]:
theta = np.array([0.0, 1.0, -0.5, 0.2, 2.0, -1.0])
beta1, beta2, sigma1, sigma2, p11, p22 = transform(theta)

print(beta1, beta2)
print(sigma1, sigma2)
print(p11, p22)

assert -1 < beta1 < 1
assert -1 < beta2 < 1
assert sigma1 > 0
assert sigma2 > 0
assert 0 < p11 < 1
assert 0 < p22 < 1

0.0 0.46211715726000974
0.6065306597126334 1.2214027581601699
0.8807970779778823 0.2689414213699951


### Density

In [100]:
def density(y_t: float, y_tm1: float, beta: float, sigma: float) -> float: 
    """
    Computing state-dependent Gaussian observation density
    """
    return 1 / (np.sqrt(2*np.pi*sigma**2)) * np.exp(-(y_t-beta*y_tm1)**2 / (2*sigma**2))


### Using it

In [101]:
y_t = 0.0
y_tm1 = 0.0
beta = 0.5
sigma1 = 1.0

val = density(y_t, y_tm1, beta, sigma1)
print(val)

assert np.isfinite(val)
assert val > 0

0.3989422804014327


### Forward

In [102]:
def forward(y: np.ndarray, beta1: float, beta2: float, sigma1: float, sigma2: float, p11: float, p22: float, pi1: float = 0.5):
    """
    Computing the scaled forward probabilities and log-likelihood
    """
    T = len(y)
    
    p12 = 1-p11
    p21 = 1-p22
    pi2 = 1-pi1

    alpha = np.zeros((T, 2))
    c = np.zeros(T)

    f1 = density(y[0], 0.0, beta1, sigma1)
    f2 = density(y[0], 0.0, beta2, sigma2)

    alpha[0, 0] = pi1 * f1
    alpha[0, 1] = pi2 * f2

    c[0] = alpha[0, 0] + alpha[0, 1]
    alpha[0, :] /= c[0]

    for t in range(1, T):
        f1 = density(y[t], y[t-1], beta1, sigma1)
        f2 = density(y[t], y[t-1], beta2, sigma2)

        alpha[t, 0] = (alpha[t-1, 0] * p11 + alpha[t-1, 1] * p21) * f1
        alpha[t, 1] = (alpha[t-1, 0] * p12 + alpha[t-1, 1] * p22) * f2

        c[t] = alpha[t, 0] + alpha[t, 1]
        alpha[t, :] /= c[t]
    
    loglik = np.sum(np.log(c))

    return alpha, c, loglik

### Using the forward algo

In [103]:
alpha, c, loglik = forward(
    y = y,
    beta1 = 0.2, 
    beta2 = 0.8,
    sigma1 = 0.5,
    sigma2 = 1.0,
    p11 = 0.95,
    p22 = 0.95
)

print(f"alpha shape: {alpha.shape}")
print(f"c shape: {c.shape}")
print(f"loglik: {loglik}")
print(f"row sum of alpha (first 5): {alpha.sum(axis=1)[:5]}")

assert alpha.shape == (T, 2)
assert c.shape == (T, )
assert np.isfinite(loglik)
assert np.allclose(alpha.sum(axis=1), 1.0)
assert np.all(c > 0)

alpha shape: (100, 2)
c shape: (100,)
loglik: -107.14377442409234
row sum of alpha (first 5): [1. 1. 1. 1. 1.]


### Negative log-likelihood

In [104]:
def negative(theta: np.ndarray, y: np.ndarray) -> float:
    """
    Computing the negative log-likelihood
    """
    beta1, beta2, sigma1, sigma2, p11, p22 = transform(theta)
    
    alpha_local, c_local, loglik = forward(
        y = y,
        beta1 = beta1,
        beta2 = beta2,
        sigma1 = sigma1, 
        sigma2 = sigma2,
        p11 = p11,
        p22 = p22,
        pi1 = 0.5
    )

    return -loglik

### Using the neg-log-likelihood

In [105]:
theta0 = np.array([0.1, 0.1, 0.0, 0.0, 2.0, 2.0])
nll = negative(theta0, y)

print(f"negative log-likelihood: {nll}")

assert np.isfinite(nll)
assert isinstance(nll, float) or np.isscalar(nll)

negative log-likelihood: 132.51750795876023


### Fit

In [106]:
def fitter(y: np.ndarray, theta0: np.ndarray, method: str = "L-BFGS-B"):
    """
    Estimatee the model parameters by maximum likelihood
    """
    result = minimize(
        fun = negative,
        x0 = theta0,
        args = (y, ), 
        method = method
    )

    beta1, beta2, sigma1, sigma2, p11, p22 = transform(result.x)

    parameters_hat = {
        "beta1" : beta1,
        "beta2" : beta2,
        "sigma1" : sigma1, 
        "sigma2" : sigma2, 
        "p11" : p11,
        "p22" : p22,
        "p12" : 1 - p11,
        "p21" : 1 - p22
    }
    return result, parameters_hat

### Using the fit

In [108]:
theta0 = np.array([0.1, 0.1, 0.0, 0.0, 2.0, 2.0])
result, parameters_hat = fitter(y, theta0)

print(f"success: {result.success}")
print(f"message: {result.message}")
print(f"estimated parameters: {parameters_hat}")

assert result.x.shape == (6, )
assert all(k in parameters_hat for k in ["beta1", "beta2", "sigma1", "sigma2", "p11", "p22"])
assert parameters_hat["sigma1"] > 0
assert parameters_hat["sigma2"] > 0
assert 0 < parameters_hat["p11"] < 1
assert 0 < parameters_hat["p22"] < 1

success: True
message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
estimated parameters: {'beta1': np.float64(0.6143421728216343), 'beta2': np.float64(0.6143421728216343), 'sigma1': np.float64(0.7492352199470326), 'sigma2': np.float64(0.7492352199470326), 'p11': np.float64(0.8807970779778823), 'p22': np.float64(0.8807970779778823), 'p12': np.float64(0.11920292202211769), 'p21': np.float64(0.11920292202211769)}
